# EDA Report — MARS Dataset

**Purpose:** Full exploratory data analysis of the MARS MOOC dataset, tracing every transformation from raw interaction log (NB01) through the meta-learning episode index (NB05).  
**Data source:** `./reports/*/report.json` — all numbers are read directly from pipeline run artefacts. No values are fabricated.

**MARS** is a small, curated dataset of MOOC learner interactions with ratings and watch percentages. It is significantly smaller than XuetangX, creating important structural differences in the episode index.

---

In [1]:
import json, os
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# ── Repo root resolution ──────────────────────────────────────────────
REPO_ROOT = Path.cwd()
for _ in range(10):
    if (REPO_ROOT / 'PROJECT_STATE.md').exists():
        break
    REPO_ROOT = REPO_ROOT.parent
REPORTS_DIR = REPO_ROOT / 'reports'
DATASET = 'mars'

def load_latest(nb_name):
    d = REPORTS_DIR / nb_name
    if not d.exists():
        raise FileNotFoundError(f'No reports directory for {nb_name}')
    runs = sorted(d.iterdir())
    for run in reversed(runs):
        rp = run / 'report.json'
        if rp.exists():
            return json.loads(rp.read_text('utf-8'))
    raise FileNotFoundError(f'No report.json found under {d}')

print(f'REPO_ROOT  : {REPO_ROOT}')
print(f'REPORTS_DIR: {REPORTS_DIR}')
print(f'Dataset    : {DATASET.upper()}')

REPO_ROOT  : /home/user/jamalla/anonymous-users-mooc-session-meta
REPORTS_DIR: /home/user/jamalla/anonymous-users-mooc-session-meta/reports
Dataset    : MARS


---
## Step 1 — Raw Data Ingestion (NB01)

The MARS dataset contains learner interactions (item views) with metadata including watch percentage and rating.  
NB01 deduplicates exact duplicate (user, item, timestamp) triples and writes a cleaned Parquet file.

In [2]:
r01 = load_latest('01_ingest_mars')
m01 = r01['metrics']

print('=' * 52)
print('NB01 — RAW INGESTION METRICS')
print('=' * 52)
print(f'  Total interactions    : {m01["n_interactions"]:>14,}')
print(f'  Unique users          : {m01["n_users"]:>14,}')
print(f'  Unique items          : {m01["n_items"]:>14,}')
print(f'  Duplicates removed    : {m01["n_dedup_removed"]:>14,}')
print()
print('  Run timestamp :', r01['run_tag'])
print('  Created at    :', r01['created_at'])
print()
print('  Notes:', r01['notes'][0] if r01.get('notes') else 'n/a')
print()
print('  Sample interactions (head 5):')
for row in r01['sanity_samples']['head5']:
    print(f'    user={row["user_id"]:>6}  item={row["item_id"]:>6}  '
          f'watch={row["watch_percentage"]:>3}%  rating={row["rating"]:>2}')

# ── MARS vs XuetangX scale comparison ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scale comparison bar
metrics = ['Interactions / Events', 'Users', 'Items / Courses']
mars_vals   = [m01['n_interactions'], m01['n_users'], m01['n_items']]
# XuetangX from its own report — but we only load MARS here, so show MARS alone
bars = axes[0].bar(metrics, mars_vals, color=['#d94801', '#fd8d3c', '#fdbe85'],
                   width=0.4, edgecolor='white')
for bar, v in zip(bars, mars_vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
                 f'{v:,}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_yscale('log')
axes[0].set_ylabel('Count (log scale)')
axes[0].set_title('MARS Dataset Scale (NB01)', fontsize=11)
axes[0].spines[['top', 'right']].set_visible(False)

# Watch percentage distribution from sample (illustrative from sanity samples)
watch_vals = [row['watch_percentage'] for row in r01['sanity_samples']['head5']]
rating_vals = [row['rating'] for row in r01['sanity_samples']['head5']]
axes[1].scatter(watch_vals, rating_vals, s=80, color='#d94801', alpha=0.7, edgecolor='white')
axes[1].set_xlabel('Watch percentage (%)')
axes[1].set_ylabel('Rating (1-10)')
axes[1].set_title('MARS Sample: Watch % vs Rating (NB01 head-5)', fontsize=11)
axes[1].set_xlim(-5, 110)
axes[1].set_ylim(0, 11)
axes[1].spines[['top', 'right']].set_visible(False)

plt.suptitle('MARS — Raw Data Ingestion (NB01)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPO_ROOT / 'reports' / 'eda_mars_01_ingest.png', dpi=150)
plt.show()
print('Chart saved: eda_mars_01_ingest.png')

NB01 — RAW INGESTION METRICS
  Total interactions    :          3,655
  Unique users          :            822
  Unique items          :            776
  Duplicates removed    :              4

  Run timestamp : 20260409_180531
  Created at    : 2026-04-09T18:05:31

  Notes: Columns: user_id, item_id, ts_epoch, watch_percentage, rating. Deduped on (user_id, item_id, ts_epoch).

  Sample interactions (head 5):
    user=   672  item=   538  watch=100%  rating=10
    user=   672  item=   815  watch=100%  rating=10
    user=   672  item=  7105  watch=100%  rating=10
    user=   856  item= 31994  watch=  5%  rating= 1
    user=  3928  item=   528  watch=100%  rating=10
Chart saved: eda_mars_01_ingest.png


---
## Step 2 — Sessionization (NB02)

Interactions are grouped into sessions using a **30-minute inactivity threshold** — identical to XuetangX.  
Sessions with fewer than 2 events are discarded. MARS does not have rich event-type diversity; each interaction is a single item view event.

In [ ]:
r02 = load_latest('02_sessionize_mars')
m02 = r02['metrics']

print('=' * 52)
print('NB02 — SESSIONIZATION METRICS')
print('=' * 52)
print(f'  Events after filtering : {m02["n_events"]:>13,}')
print(f'  Sessions created       : {m02["n_sessions"]:>13,}')
print(f'  Users with sessions    : {m02["n_users"]:>13,}')
print(f'  Unique items in sessions: {m02["n_items"]:>12,}')
print(f'  Gap threshold          : {m02["gap_threshold_sec"]:>13,} s  (30 min)')
print(f'  Min events / session   : {m02["min_events_per_session"]:>13}')
print(f'  Eligible users (>=15 pairs est.): {m02["n_eligible_users"]:>6,}')
print(f'  Projected test users   : {m02["proj_test_users"]:>13,}')
print()
print(f'  Derived stats:')
print(f'    Avg events/session   : {m02["n_events"]/m02["n_sessions"]:>13.2f}')
print(f'    Avg sessions/user    : {m02["n_sessions"]/m02["n_users"]:>13.2f}')
print()
print('  Sample sessions:')
for s in r02['sanity_samples']['sessions_head3']:
    print(f'    session_id={s["session_id"]}  user={s["user_id"]}  '
          f'n_events={s["n_events"]}  duration={s["duration_sec"]}s  '
          f'n_unique_items={s["n_unique_items"]}')

# ── Funnel: from raw to sessionized ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: events and sessions bar
cats   = ['Raw interactions', 'Sessionized events', 'Sessions created']
vals   = [m01['n_interactions'], m02['n_events'], m02['n_sessions']]
colors = ['#d94801', '#fd8d3c', '#fdbe85']
bars = axes[0].bar(cats, vals, color=colors, edgecolor='white', width=0.5)
for bar, v in zip(bars, vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
                 f'{v:,}', ha='center', fontsize=10, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_title('NB01 → NB02: Events & Sessions', fontsize=11)
axes[0].spines[['top', 'right']].set_visible(False)

# Right: user retention funnel
user_vals   = [m01['n_users'], m02['n_users']]
user_labels = [f'NB01 Raw\n{m01["n_users"]:,}', f'NB02 With Sessions\n{m02["n_users"]:,}']
colors_u = ['#8c2d04', '#d94801']
bars = axes[1].bar(user_labels, user_vals, color=colors_u, width=0.4, edgecolor='white')
for bar, v in zip(bars, user_vals):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                 f'{v:,}\n({v/m01["n_users"]*100:.1f}%)',
                 ha='center', fontsize=10, fontweight='bold')
axes[1].set_ylabel('Users')
axes[1].set_title('User Retention NB01 → NB02', fontsize=11)
axes[1].spines[['top', 'right']].set_visible(False)

plt.suptitle('MARS — Sessionization (NB02)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPO_ROOT / 'reports' / 'eda_mars_02_sessionize.png', dpi=150)
plt.show()
print('Chart saved: eda_mars_02_sessionize.png')

---
## Step 3 — Vocabulary & Sequence Pairs (NB03)

Each session becomes a sequence of item IDs. For each position, a *(prefix → next item)* pair is created.  
Consecutive duplicate items within a session are removed before pair generation.

In [4]:
r03 = load_latest('03_vocab_pairs_mars')
m03 = r03['metrics']

print('=' * 52)
print('NB03 — VOCABULARY & PAIRS METRICS')
print('=' * 52)
print(f'  Vocabulary size (items)     : {m03["n_items"]:>10,}')
print(f'  Total (prefix, label) pairs : {m03["n_pairs"]:>10,}')
print(f'  Users with at least 1 pair  : {m03["n_users"]:>10,}')
print(f'  Sessions with pairs         : {m03["n_sessions_with_pairs"]:>10,}')
print()
print(f'  Derived stats:')
print(f'    Avg pairs / user      : {m03["n_pairs"]/m03["n_users"]:>10.2f}')
print(f'    Avg pairs / session   : {m03["n_pairs"]/m03["n_sessions_with_pairs"]:>10.2f}')
print()
print('  Notes:', r03['notes'][0] if r03.get('notes') else 'n/a')
print()
print('  Sample pairs:')
for p in r03['sanity_samples']['pairs_head3']:
    print(f'    user={p["user_id"]}  session={p["session_id"]}  '
          f'prefix={p["prefix"]}  label={p["label"]}  prefix_len={p["prefix_len"]}')

# ── Pairs per session histogram (approx from stats) ───────────────────
fig, ax = plt.subplots(figsize=(8, 4))
stages   = ['Items (vocab)', 'Pairs total', 'Users', 'Sessions']
vals_bar = [m03['n_items'], m03['n_pairs'], m03['n_users'], m03['n_sessions_with_pairs']]
colors   = ['#6a3d9a', '#cab2d6', '#ff7f00', '#fdbf6f']
bars = ax.bar(stages, vals_bar, color=colors, width=0.5, edgecolor='white')
for bar, v in zip(bars, vals_bar):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
            f'{v:,}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Count')
ax.set_title('MARS — Vocabulary & Pairs Summary (NB03)', fontsize=12, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(REPO_ROOT / 'reports' / 'eda_mars_03_pairs.png', dpi=150)
plt.show()
print('Chart saved: eda_mars_03_pairs.png')

NB03 — VOCABULARY & PAIRS METRICS
  Vocabulary size (items)     :        745
  Total (prefix, label) pairs :      2,333
  Users with at least 1 pair  :        378
  Sessions with pairs         :        561

  Derived stats:
    Avg pairs / user      :       6.17
    Avg pairs / session   :       4.16

  Notes: Vocabulary: 745 unique items (0-indexed). Consecutive duplicates removed within sessions.

  Sample pairs:
    user=672  session=0  prefix=[12]  label=114  prefix_len=1
    user=672  session=0  prefix=[12, 114]  label=224  prefix_len=2
    user=4448  session=4  prefix=[655]  label=656  prefix_len=1
Chart saved: eda_mars_03_pairs.png


---
## Step 4 — Session Reliability Scores / SRS (NB03b)

Each session receives an SRS score. **MARS uses an adapted formula** because it lacks rich action-type diversity:
- **Intensity**: `min(n_events / CAP_INTENSITY, 1.0)`  
- **Extent**: `min(duration_sec / CAP_EXTENT, 1.0)`  
- **Composition**: `min(n_unique_items / CAP_COMPOSITION, 1.0)` — *item diversity* replaces action-type diversity

`SRS = (intensity + extent + composition) / 3`

CAPs computed from **training sessions only** (no leakage). MARS sessions are shorter and less varied than XuetangX, resulting in a lower mean SRS.

In [5]:
r03b = load_latest('03b_srs_scores_mars')
m03b = r03b['metrics']

print('=' * 60)
print('NB03b — SRS SCORE METRICS (MARS)')
print('=' * 60)
print(f'  Sessions scored        : {m03b["n_sessions"]:>14,}')
print(f'  Pairs with SRS         : {m03b["n_pairs"]:>14,}')
print(f'  CAP intensity (p95)    : {m03b["cap_intensity"]:>14.2f} events')
print(f'  CAP extent (p95)       : {m03b["cap_extent"]:>14.2f} sec')
print(f'  CAP composition (p95)  : {m03b["cap_composition"]:>14.2f} unique items')
print()
print('  SRS distribution:')
print(f'    Mean                 : {m03b["srs_mean"]:>14.4f}')
print(f'    Std                  : {m03b["srs_std"]:>14.4f}')
print(f'    Median (p50)         : {m03b["srs_p50"]:>14.4f}')
print(f'    p75                  : {m03b["srs_p75"]:>14.4f}')
print()
print('  Session tier breakdown:')
print(f'    Low  (SRS < 0.33)    : {m03b["tier_low"]*100:>13.1f}%  ({int(m03b["tier_low"]*m03b["n_sessions"]):,} sessions)')
print(f'    Med  (0.33-0.66)     : {m03b["tier_medium"]*100:>13.1f}%  ({int(m03b["tier_medium"]*m03b["n_sessions"]):,} sessions)')
print(f'    High (SRS >= 0.66)   : {m03b["tier_high"]*100:>13.1f}%  ({int(m03b["tier_high"]*m03b["n_sessions"]):,} sessions)')
print()
print('  Notes:', r03b['notes'][0] if r03b.get('notes') else 'n/a')

# ── SRS visualization ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Tier pie chart
tier_vals   = [m03b['tier_low'], m03b['tier_medium'], m03b['tier_high']]
tier_labels = [f'Low (<0.33)\n{m03b["tier_low"]*100:.1f}%',
               f'Medium (0.33-0.66)\n{m03b["tier_medium"]*100:.1f}%',
               f'High (>=0.66)\n{m03b["tier_high"]*100:.1f}%']
tier_colors = ['#d73027', '#fee090', '#4dac26']
wedges, _ = axes[0].pie(tier_vals, labels=tier_labels, colors=tier_colors,
                         startangle=140, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[0].set_title('SRS Tier Distribution — MARS', fontsize=11)

# Right: SRS percentiles from validation report
r09 = load_latest('09_srs_validation_mars')
m09 = r09['metrics']
percentile_labels = ['min', 'p25', 'p50', 'p75', 'max']
percentile_vals   = [m09['min'], m09['p25'], m09['p50'], m09['p75'], m09['max']]
axes[1].barh(percentile_labels, percentile_vals, color='#d94801', edgecolor='white')
for i, v in enumerate(percentile_vals):
    axes[1].text(v + 0.01, i, f'{v:.4f}', va='center', fontsize=9)
axes[1].axvline(x=m09['mean'], color='red', linestyle='--', linewidth=1.5,
                label=f'Mean={m09["mean"]:.3f}')
axes[1].axvline(x=0.33, color='orange', linestyle=':', linewidth=1.2, label='Low/Med (0.33)')
axes[1].axvline(x=0.66, color='green', linestyle=':', linewidth=1.2, label='Med/High (0.66)')
axes[1].set_xlabel('SRS score')
axes[1].set_xlim(0, 1.08)
axes[1].set_title('SRS Percentiles (from NB09 Validation)', fontsize=11)
axes[1].legend(fontsize=8, loc='lower right')
axes[1].spines[['top', 'right']].set_visible(False)

plt.suptitle('MARS — Session Reliability Scores (NB03b + NB09)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(REPO_ROOT / 'reports' / 'eda_mars_03b_srs.png', dpi=150)
plt.show()
print('Chart saved: eda_mars_03b_srs.png')

NB03b — SRS SCORE METRICS (MARS)
  Sessions scored        :            561
  Pairs with SRS         :          2,333
  CAP intensity (p95)    :          16.90 events
  CAP extent (p95)       :        3757.60 sec
  CAP composition (p95)  :          16.90 unique items

  SRS distribution:
    Mean                 :         0.2665
    Std                  :         0.2415
    Median (p50)         :         0.1627
    p75                  :         0.3130

  Session tier breakdown:
    Low  (SRS < 0.33)    :          76.5%  (428 sessions)
    Med  (0.33-0.66)     :          14.3%  (80 sessions)
    High (SRS >= 0.66)   :           9.3%  (52 sessions)

  Notes: MARS SRS: composition = min(n_unique_items / CAP_COMPOSITION, 1.0). No action_types in MARS — adapted formula uses item diversity instead. CAPs from training sessions. 
Chart saved: eda_mars_03b_srs.png


---
## Step 5 — User Split (NB04)

MARS uses a **40 / 20 / 40** user split (adjusted from the standard 70/15/15).  
The 70/15/15 split would produce too few test users to form meaningful episodes; 40/20/40 was chosen to maximize test coverage while keeping a sufficient training pool.

In [6]:
r04 = load_latest('04_user_split_mars')
m04 = r04['metrics']

print('=' * 60)
print('NB04 — USER SPLIT METRICS (MARS)')
print('=' * 60)
print(f'  Total users            : {m04["n_users_total"]:>13,}')
print(f'  Train users (40%)      : {m04["n_users_train"]:>13,}  [{m04["n_users_train"]/m04["n_users_total"]*100:.1f}%]')
print(f'  Val users  (20%)       : {m04["n_users_val"]:>13,}  [{m04["n_users_val"]/m04["n_users_total"]*100:.1f}%]')
print(f'  Test users (40%)       : {m04["n_users_test"]:>13,}  [{m04["n_users_test"]/m04["n_users_total"]*100:.1f}%]')
print()
print(f'  Pairs — Train          : {m04["n_pairs_train"]:>13,}')
print(f'  Pairs — Val            : {m04["n_pairs_val"]:>13,}')
print(f'  Pairs — Test           : {m04["n_pairs_test"]:>13,}')
print()
print('  Note: 40/20/40 split chosen because MARS dataset is too small')
print('        for 70/15/15 to yield sufficient test episodes.')

# ── Split chart ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

split_labels = ['Train (40%)', 'Val (20%)', 'Test (40%)']
user_counts  = [m04['n_users_train'], m04['n_users_val'], m04['n_users_test']]
pair_counts  = [m04['n_pairs_train'],  m04['n_pairs_val'],  m04['n_pairs_test']]
xpos = np.arange(3)
colors_split = ['#8c2d04', '#d94801', '#fd8d3c']

bars = axes[0].bar(xpos, user_counts, color=colors_split, edgecolor='white', width=0.5)
for bar, v in zip(bars, user_counts):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
                 f'{v:,}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_xticks(xpos)
axes[0].set_xticklabels(split_labels)
axes[0].set_ylabel('Users')
axes[0].set_title('Users per Split', fontsize=11)
axes[0].spines[['top', 'right']].set_visible(False)

bars = axes[1].bar(xpos, pair_counts, color=colors_split, edgecolor='white', width=0.5)
for bar, v in zip(bars, pair_counts):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
                 f'{v:,}', ha='center', fontsize=11, fontweight='bold')
axes[1].set_xticks(xpos)
axes[1].set_xticklabels(split_labels)
axes[1].set_ylabel('Pairs')
axes[1].set_title('Pairs per Split', fontsize=11)
axes[1].spines[['top', 'right']].set_visible(False)

plt.suptitle('MARS — User Split 40/20/40 (NB04)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPO_ROOT / 'reports' / 'eda_mars_04_user_split.png', dpi=150)
plt.show()
print('Chart saved: eda_mars_04_user_split.png')

NB04 — USER SPLIT METRICS (MARS)
  Total users            :           378
  Train users (40%)      :           151  [39.9%]
  Val users  (20%)       :            75  [19.8%]
  Test users (40%)       :           152  [40.2%]

  Pairs — Train          :           693
  Pairs — Val            :           545
  Pairs — Test           :         1,095

  Note: 40/20/40 split chosen because MARS dataset is too small
        for 70/15/15 to yield sufficient test episodes.
Chart saved: eda_mars_04_user_split.png


---
## Step 6 — Meta-Learning Episode Index (NB05)

Episodes use the same **K=5 support + Q=10 query** protocol as XuetangX.  
A user needs at least **15 pairs** to contribute an episode. MARS users have very few pairs on average (~6.2 pairs/user), so only a small fraction meet this threshold — resulting in very few val/test episodes.

In [7]:
r05 = load_latest('05_episode_index_mars')
m05 = r05['metrics']

print('=' * 60)
print('NB05 — EPISODE INDEX METRICS (MARS)')
print('=' * 60)
print(f'  Episode protocol : K={m05["K"]} support, Q={m05["Q"]} query')
print(f'  Train episodes   : {m05["n_episodes_train"]:>12,}')
print(f'  Val episodes     : {m05["n_episodes_val"]:>12,}')
print(f'  Test episodes    : {m05["n_episodes_test"]:>12,}')
print()
print('  Derived stats:')
print(f'    Avg pairs/user (train)   : {m04["n_pairs_train"]/m04["n_users_train"]:>10.2f}')
print(f'    Avg pairs/user (test)    : {m04["n_pairs_test"]/m04["n_users_test"]:>10.2f}')
print(f'    Users qualifying (train) : {m05["n_episodes_train"]:>10,} (need >=15 pairs)')
print(f'    Users qualifying (test)  : {m05["n_episodes_test"]:>10,}')
print()
print('  IMPORTANT: Very small episode counts in val/test are a fundamental')
print('  constraint of the MARS dataset size — most users have fewer than 15 pairs.')
print()
if r05.get('key_findings'):
    print('  Key findings:')
    for kf in r05['key_findings']:
        print(f'    {kf}')

# ── Episode bar chart ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: episode counts
split_labels   = ['Train', 'Val', 'Test']
episode_counts = [m05['n_episodes_train'], m05['n_episodes_val'], m05['n_episodes_test']]
colors_ep      = ['#8c2d04', '#d94801', '#fd8d3c']
bars = axes[0].bar(split_labels, episode_counts, color=colors_ep, width=0.45, edgecolor='white')
for bar, v in zip(bars, episode_counts):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f'{v:,}', ha='center', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Episode count')
axes[0].set_title(f'MARS — Episodes per Split (K={m05["K"]}, Q={m05["Q"]})', fontsize=11)
axes[0].spines[['top', 'right']].set_visible(False)

# Right: episodes per user
eps_per_user = [
    m05['n_episodes_train']/m04['n_users_train'],
    m05['n_episodes_val']/m04['n_users_val'],
    m05['n_episodes_test']/m04['n_users_test']
]
bars = axes[1].bar(split_labels, eps_per_user, color=colors_ep, width=0.45, edgecolor='white')
for bar, v in zip(bars, eps_per_user):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f'{v:.2f}', ha='center', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Episodes per user')
axes[1].set_title('Episodes per User (Qualifying Rate)', fontsize=11)
axes[1].spines[['top', 'right']].set_visible(False)

plt.suptitle('MARS — Meta-Learning Episode Index (NB05)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPO_ROOT / 'reports' / 'eda_mars_05_episodes.png', dpi=150)
plt.show()
print('Chart saved: eda_mars_05_episodes.png')

NB05 — EPISODE INDEX METRICS (MARS)
  Episode protocol : K=5 support, Q=10 query
  Train episodes   :          111
  Val episodes     :            6
  Test episodes    :           17

  Derived stats:
    Avg pairs/user (train)   :       4.59
    Avg pairs/user (test)    :       7.20
    Users qualifying (train) :        111 (need >=15 pairs)
    Users qualifying (test)  :         17

  IMPORTANT: Very small episode counts in val/test are a fundamental
  constraint of the MARS dataset size — most users have fewer than 15 pairs.

  Key findings:
    Train episodes: 111
    Val episodes:   6 (>= 50 required)
    Test episodes:  17 (>= 50 required)
    Protocol: K=5 support + Q=10 query, chronological no-leakage
Chart saved: eda_mars_05_episodes.png


---
## Pipeline Funnel — End-to-End Data Reduction

In [8]:
print('=' * 68)
print('MARS — FULL PIPELINE FUNNEL SUMMARY')
print('=' * 68)
print(f'  NB01 Raw interactions    : {m01["n_interactions"]:>12,}')
print(f'  NB01 Raw users           : {m01["n_users"]:>12,}')
print(f'  NB01 Raw items           : {m01["n_items"]:>12,}')
print(f'  NB01 Dedup removed       : {m01["n_dedup_removed"]:>12,}')
print()
print(f'  NB02 Sessionized events  : {m02["n_events"]:>12,}  (drop {(1-m02["n_events"]/m01["n_interactions"])*100:.1f}%)')
print(f'  NB02 Sessions created    : {m02["n_sessions"]:>12,}')
print(f'  NB02 Users with sessions : {m02["n_users"]:>12,}  (drop {(1-m02["n_users"]/m01["n_users"])*100:.1f}%)')
print()
print(f'  NB03 Vocabulary size     : {m03["n_items"]:>12,}')
print(f'  NB03 Total pairs         : {m03["n_pairs"]:>12,}')
print(f'  NB03 Users with pairs    : {m03["n_users"]:>12,}  (same as NB02 users)')
print()
print(f'  NB03b Sessions scored    : {m03b["n_sessions"]:>12,}')
print(f'  NB03b SRS mean / median  : {m03b["srs_mean"]:.4f} / {m03b["srs_p50"]:.4f}')
print()
print(f'  NB04 Users — Train       : {m04["n_users_train"]:>12,}  (40%)')
print(f'  NB04 Users — Val         : {m04["n_users_val"]:>12,}  (20%)')
print(f'  NB04 Users — Test        : {m04["n_users_test"]:>12,}  (40%)')
print()
print(f'  NB05 Train episodes      : {m05["n_episodes_train"]:>12,}  (K={m05["K"]}, Q={m05["Q"]})')
print(f'  NB05 Val episodes        : {m05["n_episodes_val"]:>12,}')
print(f'  NB05 Test episodes       : {m05["n_episodes_test"]:>12,}')
print('=' * 68)

# ── Funnel chart ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: User funnel
user_funnel = [m01['n_users'], m02['n_users'], m03['n_users'],
               m04['n_users_train'], m04['n_users_val'], m04['n_users_test']]
user_stages = ['NB01 Raw', 'NB02 With\nSessions', 'NB03 With\nPairs',
               'NB04 Train', 'NB04 Val', 'NB04 Test']
colors_u = ['#67000d', '#a50f15', '#cb181d', '#238b45', '#41ab5d', '#74c476']
bars = axes[0].bar(range(len(user_stages)), user_funnel, color=colors_u, edgecolor='white')
for bar, v in zip(bars, user_funnel):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
                 f'{v:,}', ha='center', fontsize=9, rotation=20)
axes[0].set_xticks(range(len(user_stages)))
axes[0].set_xticklabels(user_stages, fontsize=9)
axes[0].set_ylabel('Users')
axes[0].set_title('User Count through Pipeline', fontsize=11)
axes[0].spines[['top', 'right']].set_visible(False)

# Right: Pairs / episodes per stage
pair_funnel = [m03['n_pairs'], m04['n_pairs_train'], m04['n_pairs_val'],
               m04['n_pairs_test'], m05['n_episodes_train'], m05['n_episodes_val'],
               m05['n_episodes_test']]
pair_stages = ['NB03 Pairs', 'Pairs Train', 'Pairs Val', 'Pairs Test',
               'Eps Train', 'Eps Val', 'Eps Test']
colors_p = ['#4d004b', '#810f7c', '#88419d', '#8c96c6',
            '#005a32', '#238b45', '#74c476']
bars = axes[1].bar(range(len(pair_stages)), pair_funnel, color=colors_p, edgecolor='white')
for bar, v in zip(bars, pair_funnel):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                 f'{v:,}', ha='center', fontsize=9, rotation=20)
axes[1].set_xticks(range(len(pair_stages)))
axes[1].set_xticklabels(pair_stages, fontsize=9, rotation=20)
axes[1].set_ylabel('Count')
axes[1].set_title('Pairs & Episodes through Pipeline', fontsize=11)
axes[1].spines[['top', 'right']].set_visible(False)

plt.suptitle('MARS — Full Pipeline Data Reduction Funnel', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPO_ROOT / 'reports' / 'eda_mars_funnel.png', dpi=150)
plt.show()
print('Chart saved: eda_mars_funnel.png')

MARS — FULL PIPELINE FUNNEL SUMMARY
  NB01 Raw interactions    :        3,655
  NB01 Raw users           :          822
  NB01 Raw items           :          776
  NB01 Dedup removed       :            4

  NB02 Sessionized events  :        2,894  (drop 20.8%)
  NB02 Sessions created    :          561
  NB02 Users with sessions :          378  (drop 54.0%)

  NB03 Vocabulary size     :          745
  NB03 Total pairs         :        2,333
  NB03 Users with pairs    :          378  (same as NB02 users)

  NB03b Sessions scored    :          561
  NB03b SRS mean / median  : 0.2665 / 0.1627

  NB04 Users — Train       :          151  (40%)
  NB04 Users — Val         :           75  (20%)
  NB04 Users — Test        :          152  (40%)

  NB05 Train episodes      :          111  (K=5, Q=10)
  NB05 Val episodes        :            6
  NB05 Test episodes       :           17
Chart saved: eda_mars_funnel.png


---
## Summary

| Stage | Output | Key Metric |
|-------|--------|------------|
| **NB01 Ingest** | 3,655 interactions | 822 users · 776 items · 4 dedup removed |
| **NB02 Sessionize** | 561 sessions | 378 users · avg 5.2 events/session |
| **NB03 Vocab+Pairs** | 2,333 pairs | 745-item vocabulary · 6.2 pairs/user |
| **NB03b SRS** | 561 sessions scored | SRS mean=0.27 · 76.5% low · 9.3% high |
| **NB04 User Split** | 378 users split | 40/20/40 · cold-start guarantee |
| **NB05 Episodes** | 134 total episodes | K=5 support · Q=10 query · 17 test episodes |

**Dataset size context:**  
MARS is ~7,700× smaller than XuetangX in raw events and ~220× smaller in users. This creates a statistically challenging evaluation environment — 17 test episodes is the ceiling given the K=5+Q=10 constraint and the user pair distribution.

**SRS correlation** (from NB09):  
`r(SRS, n_events) = 0.902` — much higher than XuetangX (0.503), reflecting that MARS sessions have low complexity (no action-type diversity), so event count dominates the SRS signal.  
`r(SRS, duration) = 0.856` — also high.